# 03 - Feature Engineering

**Proyecto:** Pochoclo Predict

## Objetivo de este notebook

Transformar el dataset limpio de `02_eda.ipynb` en una matriz de features
numerica, lista para entrenar modelos de regresion sobre `nota_promedio`.
Se aplican tecnicas de **Web / Text Mining** vistas en la materia:

- **TF-IDF** sobre el reparto principal (elenco), para capturar el "peso" de
  actores que aparecen frecuentemente en el dataset ("star power"), reducido
  con **SVD** (analogo a LSA) a un puñado de componentes densas.
- **TF-IDF** sobre palabras clave (`keywords`), con el mismo tratamiento.
- **Analisis de sentimiento** sobre la sinopsis, con el lexicon **VADER**.
- **Codificacion de generos**: el genero principal como variable categorica
  (one-hot) y los subgeneros restantes como variables **multi-etiqueta**
  (multi-label), ya que una pelicula puede pertenecer a varios generos a la
  vez ademas del principal.


In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

import json

import numpy as np
import pandas as pd
import nltk
from sklearn.decomposition import TruncatedSVD
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import MultiLabelBinarizer

from src import paths

pd.set_option("display.max_columns", None)

nltk.download("vader_lexicon", quiet=True)
from nltk.sentiment import SentimentIntensityAnalyzer


In [2]:
df = pd.read_csv(paths.EDA_CLEAN_CSV, encoding="utf-8-sig")

list_cols = ["generos", "companias_productoras", "paises_produccion", "reparto_principal", "keywords"]
for col in list_cols:
    df[col] = df[col].apply(lambda x: json.loads(x) if isinstance(x, str) else [])

print(f"Dataset de entrada: {df.shape[0]} filas x {df.shape[1]} columnas")


Dataset de entrada: 2998 filas x 28 columnas


## 1. Variables numericas base

Se aplica `log1p` a las variables con distribucion muy asimetrica
(identificadas en el EDA) y se codifica el mes de estreno de forma **ciclica**
(`sin`/`cos`) para que el modelo entienda que diciembre y enero estan
"cerca" en el calendario, algo que una codificacion numerica lineal (1-12)
no puede representar.


In [3]:
feat = pd.DataFrame(index=df.index)
feat["id"] = df["id"]
feat["nota_promedio"] = df["nota_promedio"]

feat["log_presupuesto"] = np.log1p(df["presupuesto"])
feat["log_recaudacion"] = np.log1p(df["recaudacion"])
feat["presupuesto_conocido"] = df["presupuesto_conocido"]
feat["recaudacion_conocido"] = df["recaudacion_conocido"]

feat["duracion_min"] = df["duracion_min"]
feat["log_popularidad"] = np.log1p(df["popularidad"])
feat["log_cantidad_votos"] = np.log1p(df["cantidad_votos"])

feat["anio_estreno"] = df["anio_estreno"]
feat["mes_sin"] = np.sin(2 * np.pi * df["mes_estreno"] / 12)
feat["mes_cos"] = np.cos(2 * np.pi * df["mes_estreno"] / 12)

feat["num_generos"] = df["num_generos"]
feat["num_companias_productoras"] = df["companias_productoras"].apply(len)
feat["num_paises_produccion"] = df["paises_produccion"].apply(len)

feat.head()


,id,nota_promedio,log_presupuesto,log_recaudacion,presupuesto_conocido,recaudacion_conocido,duracion_min,log_popularidad,log_cantidad_votos,anio_estreno,mes_sin,mes_cos,num_generos,num_companias_productoras,num_paises_produccion
0,157336,8.488,18.921456,20.431049,1,1,169,4.289552,10.627309,2014,-5.000000e-01,0.866025,3,3,2
1,27205,8.374,18.890684,20.547758,1,1,148,4.107792,10.603014,2010,-5.000000e-01,-0.866025,3,1,2
2,24428,8.077,19.209138,21.141197,1,1,142,4.168389,10.592024,2012,8.660254e-01,-0.500000,3,1,1
3,155,8.535,19.035866,20.727814,1,1,152,4.171134,10.512927,2008,-5.000000e-01,-0.866025,3,4,2
4,19995,7.609,19.283571,21.796118,1,1,161,3.866588,10.455042,2009,-2.449294e-16,1.000000,3,4,2


## 2. Experiencia del director

Con alta cardinalidad (cientos de directores distintos) no tiene sentido un
one-hot. En cambio se construye una variable proxy de **reputacion/experiencia**:
la cantidad de peliculas del mismo director presentes en el propio dataset.
Los directores desconocidos (sin dato) reciben el valor minimo (1).


In [4]:
conteo_director = df["director"].value_counts()
feat["director_experiencia"] = df["director"].map(conteo_director).fillna(1).astype(int)


## 3. Generos: principal (one-hot) + subgeneros (multi-label)


In [5]:
genero_principal_dummies = pd.get_dummies(df["genero_principal"], prefix="genero")
feat = pd.concat([feat, genero_principal_dummies], axis=1)

subgeneros = df["generos"].apply(lambda gs: gs[1:] if len(gs) > 1 else [])
mlb = MultiLabelBinarizer()
subgeneros_bin = pd.DataFrame(
    mlb.fit_transform(subgeneros),
    columns=[f"subgenero_{g}" for g in mlb.classes_],
    index=df.index,
)
feat = pd.concat([feat, subgeneros_bin], axis=1)

print(f"Generos principales (one-hot): {genero_principal_dummies.shape[1]} columnas")
print(f"Subgeneros (multi-label): {subgeneros_bin.shape[1]} columnas")


Generos principales (one-hot): 18 columnas
Subgeneros (multi-label): 18 columnas


## 4. TF-IDF del reparto principal (actores)

Cada pelicula se representa como un "documento" cuyos tokens son sus actores
principales (se reemplaza el espacio interno del nombre por `_` para que
cada actor sea un unico token, ej. `Matthew_McConaughey`). TF-IDF le da mas
peso a actores que aparecen en **varias** peliculas del dataset (frecuentes
pero no universales) frente a apariciones unicas. La matriz resultante,
dispersa y de alta dimension, se reduce con **SVD** a un numero chico de
componentes densas que resumen el "perfil de elenco" de cada pelicula.


In [6]:
def a_documento(lista_actores):
    return " ".join(a.replace(" ", "_") for a in lista_actores)

docs_actores = df["reparto_principal"].apply(a_documento)

tfidf_actores = TfidfVectorizer(max_features=500, min_df=2, token_pattern=r"[^\s]+")
matriz_actores = tfidf_actores.fit_transform(docs_actores)

N_COMPONENTES_ACTORES = 25
svd_actores = TruncatedSVD(n_components=N_COMPONENTES_ACTORES, random_state=42)
actores_svd = svd_actores.fit_transform(matriz_actores)

actores_svd_df = pd.DataFrame(
    actores_svd, columns=[f"actor_svd_{i+1}" for i in range(N_COMPONENTES_ACTORES)], index=df.index
)
feat = pd.concat([feat, actores_svd_df], axis=1)

print(f"Vocabulario de actores (TF-IDF): {len(tfidf_actores.vocabulary_)} actores distintos")
print(f"Varianza explicada por los {N_COMPONENTES_ACTORES} componentes SVD: {svd_actores.explained_variance_ratio_.sum():.1%}")


Vocabulario de actores (TF-IDF): 500 actores distintos
Varianza explicada por los 25 componentes SVD: 11.6%


## 5. TF-IDF de palabras clave (keywords)

Mismo tratamiento que con el elenco, pero sobre las `keywords` que TMDB
asigna a cada pelicula (temas, objetos, tropos narrativos). Estas palabras
clave suelen ser mas informativas del *contenido* de la pelicula que el
genero, que es una categoria mucho mas amplia.


In [7]:
def kw_a_documento(lista_kw):
    return " ".join(k.replace(" ", "_") for k in lista_kw)

docs_kw = df["keywords"].apply(kw_a_documento)

tfidf_kw = TfidfVectorizer(max_features=500, min_df=3, token_pattern=r"[^\s]+")
matriz_kw = tfidf_kw.fit_transform(docs_kw)

N_COMPONENTES_KW = 20
svd_kw = TruncatedSVD(n_components=N_COMPONENTES_KW, random_state=42)
kw_svd = svd_kw.fit_transform(matriz_kw)

kw_svd_df = pd.DataFrame(
    kw_svd, columns=[f"keyword_svd_{i+1}" for i in range(N_COMPONENTES_KW)], index=df.index
)
feat = pd.concat([feat, kw_svd_df], axis=1)

print(f"Vocabulario de keywords (TF-IDF): {len(tfidf_kw.vocabulary_)} palabras distintas")
print(f"Varianza explicada por los {N_COMPONENTES_KW} componentes SVD: {svd_kw.explained_variance_ratio_.sum():.1%}")


Vocabulario de keywords (TF-IDF): 500 palabras distintas
Varianza explicada por los 20 componentes SVD: 15.2%


## 6. Analisis de sentimiento de la sinopsis

Se utiliza **VADER** (Valence Aware Dictionary and sEntiment Reasoner), un
analizador de sentimiento basado en lexicon, validado y ampliamente usado
como baseline en la literatura de NLP. VADER esta calibrado para **texto en
ingles**; por eso se aplica sobre `sinopsis_en` (la traduccion al ingles
obtenida de la propia API de TMDB en el NB01) en lugar de la sinopsis en
espanol, evitando así la necesidad de un lexicon o modelo especifico para
espanol fuera del alcance de este trabajo.


In [8]:
sia = SentimentIntensityAnalyzer()

def sentimiento(texto):
    texto = texto if isinstance(texto, str) and texto.strip() else ""
    if not texto:
        return pd.Series({"sentimiento_compound": 0.0, "sentimiento_pos": 0.0,
                           "sentimiento_neg": 0.0, "sentimiento_neu": 1.0})
    s = sia.polarity_scores(texto)
    return pd.Series({"sentimiento_compound": s["compound"], "sentimiento_pos": s["pos"],
                       "sentimiento_neg": s["neg"], "sentimiento_neu": s["neu"]})

sentimiento_df = df["sinopsis_en"].apply(sentimiento)
feat = pd.concat([feat, sentimiento_df], axis=1)
feat["sinopsis_longitud"] = df["sinopsis_es"].fillna("").apply(lambda t: len(t.split()))

sentimiento_df.describe()


,sentimiento_compound,sentimiento_pos,sentimiento_neg,sentimiento_neu
count,2998.000000,2998.000000,2998.000000,2998.000000
mean,-0.126041,0.099266,0.128781,0.771951
std,0.619337,0.082961,0.098425,0.110712
min,-0.987700,0.000000,0.000000,0.378000
25%,-0.709600,0.033000,0.055000,0.699000
50%,-0.226300,0.090000,0.116000,0.774000
75%,0.440400,0.148000,0.189000,0.849000
max,0.989800,0.470000,0.531000,1.000000


## 7. Dataset final de features


In [9]:
print(f"Shape final: {feat.shape[0]} filas x {feat.shape[1]} columnas")
print(f"Nulos remanentes: {feat.isna().sum().sum()}")

paths.DATA_PROCESSED.mkdir(parents=True, exist_ok=True)
feat.to_csv(paths.FEATURES_CSV, index=False, encoding="utf-8-sig")
print(f"Guardado en: {paths.FEATURES_CSV.relative_to(paths.PROJECT_ROOT)}")


Shape final: 2998 filas x 102 columnas
Nulos remanentes: 0


Guardado en: data\processed\dataset_features.csv


In [10]:
correlaciones = feat.drop(columns=["id"]).corr(numeric_only=True)["nota_promedio"].drop("nota_promedio")
correlaciones = correlaciones.reindex(correlaciones.abs().sort_values(ascending=False).index)
correlaciones.head(20)


log_cantidad_votos           0.340939
duracion_min                 0.317834
log_popularidad              0.300808
genero_Drama                 0.269596
anio_estreno                -0.223992
keyword_svd_2                0.220306
log_presupuesto             -0.209221
subgenero_Drama              0.164968
keyword_svd_18               0.139031
genero_Acción               -0.132038
keyword_svd_20               0.130224
actor_svd_9                 -0.129870
genero_Comedia              -0.129217
genero_Terror               -0.126083
keyword_svd_7               -0.124722
subgenero_Historia           0.124442
keyword_svd_3                0.118975
genero_Animación             0.116906
subgenero_Terror            -0.116552
num_companias_productoras   -0.107446
Name: nota_promedio, dtype: float64